# Retail Demand & Basket Analysis\nArrowstack Data Science Internship — Project 2\n\n**Objective:** identify demand patterns and product associations that a merchandising stakeholder can evaluate.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom mlxtend.frequent_patterns import apriori, association_rules\n\ncandidates = [Path('data/retail_transactions.csv'), Path('../data/retail_transactions.csv')]\nDATA_PATH = next((p for p in candidates if p.exists()), None)\nif DATA_PATH is None: raise FileNotFoundError('Could not locate data/retail_transactions.csv')\ndf = pd.read_csv(DATA_PATH, parse_dates=['transaction_date'])\ndf.head()

## 1. Data quality checks\nValidate uniqueness, missingness, date parsing, allowed segments, basket-size consistency and product counts before analysis.

In [ ]:
allowed_segments = {'Value','Regular','Premium'}\nparsed = df['items'].str.split('|')\nqa = {\n    'rows': len(df),\n    'duplicate_ids': int(df['transaction_id'].duplicated().sum()),\n    'missing_cells': int(df.isna().sum().sum()),\n    'invalid_segments': int((~df['customer_segment'].isin(allowed_segments)).sum()),\n    'basket_size_mismatch': int((parsed.str.len() != df['basket_size']).sum()),\n    'invalid_basket_size': int((~df['basket_size'].between(2,8)).sum()),\n}\npd.Series(qa)

## 2. Business question: demand concentration\nMeasure product coverage across transactions and basket-size distribution.

In [ ]:
item_counts = {}\nfor items in parsed:\n    for item in items: item_counts[item] = item_counts.get(item, 0) + 1\nitem_summary = (pd.Series(item_counts, name='transactions').sort_values(ascending=False)\n               .to_frame().assign(support=lambda x: x['transactions']/len(df)))\ndisplay(item_summary.head(10))\ndf['basket_size'].describe()

In [ ]:
item_summary.head(10)['support'].sort_values().plot(kind='barh', title='Top Product Transaction Support')\nplt.xlabel('Support (share of transactions)'); plt.tight_layout(); plt.show()

## 3. Business question: which products are associated?\nCreate a transaction-item matrix, mine frequent itemsets and calculate support, confidence and lift.

In [ ]:
items = sorted({item for basket in parsed for item in basket})\nbasket_matrix = pd.DataFrame(False, index=df['transaction_id'], columns=items)\nfor tid, basket in zip(df['transaction_id'], parsed): basket_matrix.loc[tid, basket] = True\nfrequent = apriori(basket_matrix, min_support=0.08, use_colnames=True)\nrules = association_rules(frequent, metric='lift', min_threshold=1.10)\nrules = rules.sort_values(['lift','support','confidence'], ascending=False)\nrules[['antecedents','consequents','support','confidence','lift']].head(15)

## 4. Evidence-based interpretation\nUse high-lift rules with meaningful support as candidates for bundle/placement experiments. Do not interpret association as causal impact.

In [ ]:
top_rules = rules.head(10).copy()\ntop_rules['antecedents'] = top_rules['antecedents'].apply(lambda s: ', '.join(sorted(s)))\ntop_rules['consequents'] = top_rules['consequents'].apply(lambda s: ', '.join(sorted(s)))\ndisplay(top_rules[['antecedents','consequents','support','confidence','lift']])

## 5. Limitations and next steps\nThe dataset is synthetic and lacks price, margin, promotion, stockout, store and time-of-day features. Add authorized real transaction data and evaluate recommendations with controlled experiments measuring incremental basket value and revenue.

In [ ]:
assert qa['duplicate_ids'] == 0\nassert qa['missing_cells'] == 0\nassert qa['invalid_segments'] == 0\nassert qa['basket_size_mismatch'] == 0\nassert qa['invalid_basket_size'] == 0\nassert len(basket_matrix) == len(df)\nprint('ANALYSIS QA: PASS')